# Task 1 — Sentiment Analysis

Train and compare a TF-IDF baseline against a fine-tuned transformer. Report accuracy, per-class precision/recall/F1 and the confusion matrix on a held-out test set. Save the chosen model to `models/`.

In [1]:
import sys
sys.path.append("..")

import importlib
import re
import string

import matplotlib.pyplot as plt
import pandas as pd
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from src import config, preprocessing

importlib.reload(preprocessing)

pre_processed = preprocessing.load_pre_processed()
pre_processed.info()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/yasaman.nj/ironhack/projects/project-brief-NLP-automated-customers-reviews/data/processed/cleaned_1429_1.csv'

In [ ]:
rating_to_sentiment = {
    1.0: "bad",
    2.0: "bad",
    3.0: "neutral",
    4.0: "good",
    5.0: "good",
}
y = pre_processed["reviews.rating"].map(rating_to_sentiment)
X = pre_processed["reviews.text"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)
# X_train, X_test, y_train, y_test = train_test_split(pre_processed['reviews.text'], pre_processed['reviews.rating'], test_size=0.2, random_state=42)

print(X_train.shape, y_train.shape)          # (22290,) (22290,)
print(X_train.value_counts(normalize=True))
print(y_train.value_counts(normalize=True))
print(type(X_train))

(19504,) (19504,)
reviews.text
Product is awesome. Amazing how I ask a question and in a few seconds she has the answer.                                                                                                                                                                                        0.000051
Brought for grandmother as a Christmas gift because she wanted and iPad and we. Thought she'd never figure out how to use one so I bought this as a beginners tablet and she loves it                                                                                            0.000051
Great tablet for the price. Add to that expandable memory and it's a great gift for the holidays.                                                                                                                                                                                0.000051
This is a great tables for browsing the net, reading or listening to music. However when moving to video apps like YouTube,

In [ ]:
import re

# Remove inline JS/CSS first
# Then remove comments before tags, because comments may contain '>'
# Finally remove the remaining HTML tags

def remove_html_tags(text):
    text = re.sub(r"<script.*?</script>", " ", text, flags=re.I | re.S)
    text = re.sub(r"<style.*?</style>", " ", text, flags=re.I | re.S)
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.I | re.S)
    text = re.sub(r"<[^>]+>", " ", text)
    return text

# Apply to the text column
X_train_cleaned = X_train.apply(remove_html_tags)
X_test_cleaned = X_test.apply(remove_html_tags)

In [ ]:
# Remove special characters, numbers, and extra spacing
# Remove single-letter words and prefixed 'b'
# Convert to lowercase

def clean_text(text):
    if not isinstance(text, str):
        text = str(text)

    text = text.lower()
    #text = text.replace("b'", "").replace('b"', "")
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\b[a-z]\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

X_train_cleaned = X_train_cleaned.apply(clean_text)
X_test_cleaned = X_test_cleaned.apply(clean_text)

X_train_cleaned.head()

26559    product is awesome amazing how ask question an...
18910    love the kindle paperwhite it is so light weig...
8037     great for kids easy to use great price would r...
4539     this was great purchase love the size and func...
8332     it is great for reading but that about it ther...
Name: reviews.text, dtype: object

In [ ]:
def remove_stopwords(text):
    stop_words = set(stopwords.words("english"))
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return " ".join(filtered_words)

X_train_cleaned = X_train_cleaned.apply(remove_stopwords)
X_test_cleaned = X_test_cleaned.apply(remove_stopwords)

X_train_cleaned.head(10)

26559    product awesome amazing ask question seconds a...
18910    love kindle paperwhite light weight best read ...
8037       great kids easy use great price would recommend
4539                great purchase love size functionality
8332     great reading ads quite annoying well slow spe...
14375    looking good cheap tablet general use fit bill...
7664     purchased gift christmas girlfriend gave extra...
1040     avid amazon user shopping tablets bought diffe...
597      thing looks nice runs nice feels nice price th...
2312                    lot cheaper tablets functions need
Name: reviews.text, dtype: object

In [ ]:
# Your code
from nltk.stem.wordnet import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
X_train_lemmatized = X_train_cleaned.apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))
X_test_lemmatized = X_test_cleaned.apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))

In [ ]:
# Convert reviews to TF-IDF features.
from sklearn.feature_extraction.text import TfidfVectorizer


tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train_lemmatized)
X_test_tfidf = tfidf.transform(X_test_lemmatized)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)


def top_tfidf_terms_for_ratings(categories, labels, tfidf_matrix, feature_names, top_n=10):
    """Return terms with the highest average TF-IDF for selected labels."""
    category_rows = labels.isin(categories).to_numpy()
    if not category_rows.any():
        raise ValueError(f"No training rows found for labels {categories}; check y_train values.")
    mean_scores = tfidf_matrix[category_rows].mean(axis=0).A1
    return pd.Series(mean_scores, index=feature_names).nlargest(top_n)


feature_names = tfidf.get_feature_names_out()
labels = y_train.reset_index(drop=True)

negative = top_tfidf_terms_for_ratings(["bad"], labels, X_train_tfidf, feature_names)
neutral = top_tfidf_terms_for_ratings(["neutral"], labels, X_train_tfidf, feature_names)
positive = top_tfidf_terms_for_ratings(["good"], labels, X_train_tfidf, feature_names)

print("\nTop negative TF-IDF terms:\n", negative.head())
print("\nTop neutral TF-IDF terms:\n", neutral.head())
print("\nTop positive TF-IDF terms:\n", positive.head())

Train TF-IDF shape: (19504, 5000)
Test TF-IDF shape: (8359, 5000)

Top negative TF-IDF terms:
 tablet    0.040975
amazon    0.038305
slow      0.032070
work      0.029584
buy       0.028488
dtype: float64

Top neutral TF-IDF terms:
 tablet    0.055714
good      0.044583
price     0.033156
like      0.029135
apps      0.026756
dtype: float64

Top positive TF-IDF terms:
 great     0.053797
love      0.050576
tablet    0.049604
use       0.036796
easy      0.035177
dtype: float64


In [ ]:
# Convert reviews to TF-IDF features.
from sklearn.feature_extraction.text import TfidfVectorizer


tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
    sublinear_tf=True,
)
X_train_tfidf = tfidf.fit_transform(X_train_lemmatized)
X_test_tfidf = tfidf.transform(X_test_lemmatized)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)


def top_tfidf_terms_for_ratings(categories, labels, tfidf_matrix, feature_names, top_n=10):
    """Return terms with the highest average TF-IDF for selected labels."""
    category_rows = labels.isin(categories).to_numpy()
    if not category_rows.any():
        raise ValueError(f"No training rows found for labels {categories}; check y_train values.")
    mean_scores = tfidf_matrix[category_rows].mean(axis=0).A1
    return pd.Series(mean_scores, index=feature_names).nlargest(top_n)


feature_names = tfidf.get_feature_names_out()
labels = y_train.reset_index(drop=True)

negative = top_tfidf_terms_for_ratings(["bad"], labels, X_train_tfidf, feature_names)
neutral = top_tfidf_terms_for_ratings(["neutral"], labels, X_train_tfidf, feature_names)
positive = top_tfidf_terms_for_ratings(["good"], labels, X_train_tfidf, feature_names)

print("\nTop negative TF-IDF terms:\n", negative.head())
print("\nTop neutral TF-IDF terms:\n", neutral.head())
print("\nTop positive TF-IDF terms:\n", positive.head())

Train TF-IDF shape: (19504, 20000)
Test TF-IDF shape: (8359, 20000)

Top negative TF-IDF terms:
 amazon      0.025565
tablet      0.024714
slow        0.023149
returned    0.019698
work        0.019071
dtype: float64

Top neutral TF-IDF terms:
 tablet    0.033078
good      0.027631
price     0.021269
like      0.018453
ok        0.017315
dtype: float64

Top positive TF-IDF terms:
 great     0.030453
love      0.028384
tablet    0.027558
use       0.021026
easy      0.020141
dtype: float64


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

class_order = ["bad", "neutral", "good"]
nb_model = MultinomialNB()
nb_sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
nb_model.fit(X_train_tfidf, y_train, sample_weight=nb_sample_weights)

y_pred_nb = nb_model.predict(X_test_tfidf)

print("Multinomial Naive Bayes")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb, labels=class_order, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_nb, labels=class_order))

Multinomial Naive Bayes
Accuracy: 0.7834669218806077
              precision    recall  f1-score   support

         bad       0.18      0.57      0.27       199
     neutral       0.15      0.53      0.23       387
        good       0.98      0.80      0.88      7773

    accuracy                           0.78      8359
   macro avg       0.44      0.63      0.46      8359
weighted avg       0.92      0.78      0.84      8359

Confusion matrix:
 [[ 113   62   24]
 [  91  204   92]
 [ 429 1112 6232]]


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

classifier = LogisticRegression(class_weight="balanced", max_iter=1000)
classifier.fit(X_train_tfidf, y_train)

y_pred = classifier.predict(X_test_tfidf)

print(classification_report(y_test, y_pred, labels=["bad", "neutral", "good"], zero_division=0))
print(confusion_matrix(y_test, y_pred, labels=["bad", "neutral", "good"]))

              precision    recall  f1-score   support

         bad       0.31      0.46      0.37       199
     neutral       0.22      0.45      0.30       387
        good       0.97      0.91      0.94      7773

    accuracy                           0.88      8359
   macro avg       0.50      0.61      0.54      8359
weighted avg       0.92      0.88      0.90      8359

[[  92   53   54]
 [  50  173  164]
 [ 155  544 7074]]


In [ ]:
from sklearn.svm import LinearSVC

model = LinearSVC(class_weight="balanced")
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

class_order = ["bad", "neutral", "good"]

print("Accuracy:", accuracy_score(y_test, y_pred))
print(
    classification_report(
        y_test,
        y_pred,
        labels=class_order,
        zero_division=0,
    )
)
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=class_order))

Accuracy: 0.924273238425649
              precision    recall  f1-score   support

         bad       0.42      0.24      0.31       199
     neutral       0.34      0.26      0.29       387
        good       0.95      0.98      0.96      7773

    accuracy                           0.92      8359
   macro avg       0.57      0.49      0.52      8359
weighted avg       0.91      0.92      0.92      8359

Confusion matrix:
[[  48   36  115]
 [  27   99  261]
 [  40  154 7579]]
